# Dairy Sales Analysis and Demand Estimation

This notebook explores transaction-level sales and compares regression models for estimating units sold. The final 20% of the date range is held out for testing so later transactions are not mixed into training.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv('../data/dairy_sales_50000.csv', parse_dates=['Date'])
display(df.head())
print('Rows and columns:', df.shape)

In [ ]:
# Check the source data before calculating any metrics.
print('Missing values by column:\n', df.isna().sum())
print('Duplicate rows:', df.duplicated().sum())
print('Date range:', df['Date'].min().date(), 'to', df['Date'].max().date())
print('Non-positive units:', (df['Units_Sold'] <= 0).sum())

# Revenue, cost and profit are already supplied in the generated dataset.
# Do not replace profit with an assumed margin.
df = df.drop_duplicates().copy()
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['DayOfWeek'] = df['Date'].dt.dayofweek
display(df[['Units_Sold', 'Unit_Price', 'Discount_Pct', 'Revenue', 'Cost', 'Profit']].describe())

In [ ]:
monthly = df.groupby(df['Date'].dt.to_period('M')).agg(
    Revenue=('Revenue', 'sum'), Units_Sold=('Units_Sold', 'sum'), Profit=('Profit', 'sum')
).reset_index()
monthly['Date'] = monthly['Date'].dt.to_timestamp()
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(monthly['Date'], monthly['Revenue'])
ax.set(title='Monthly revenue', xlabel='Month', ylabel='Revenue')
fig.tight_layout()
plt.show()

product = df.groupby(['Product', 'Category']).agg(
    Units_Sold=('Units_Sold', 'sum'), Revenue=('Revenue', 'sum'), Profit=('Profit', 'sum')
).sort_values('Revenue', ascending=False)
region = df.groupby('Region').agg(
    Units_Sold=('Units_Sold', 'sum'), Revenue=('Revenue', 'sum'), Profit=('Profit', 'sum')
).sort_values('Revenue', ascending=False)
display(product.head(10))
display(region)

In [ ]:
df['Discount_Band'] = pd.cut(
    df['Discount_Pct'], bins=[-0.1, 4, 9, 14, 100],
    labels=['0–4%', '4–9%', '9–14%', '14%+'], include_lowest=True
)
discount_summary = df.groupby('Discount_Band', observed=False).agg(
    Transactions=('Date', 'size'), Units_Sold=('Units_Sold', 'sum'),
    Revenue=('Revenue', 'sum'), Profit=('Profit', 'sum')
)
display(discount_summary)

## Demand estimation

The target is transaction-level `Units_Sold`. A chronological holdout tests performance on later dates. This is a regression exercise, not a full inventory forecasting system: the dataset does not contain stock levels, stock-outs, or historical lag features.

In [ ]:
features = ['Year', 'Month', 'DayOfWeek', 'Product', 'Category', 'Region', 'Channel', 'Customer_Type', 'Unit_Price', 'Discount_Pct']
target = 'Units_Sold'
categorical = ['Product', 'Category', 'Region', 'Channel', 'Customer_Type']
numeric = ['Year', 'Month', 'DayOfWeek', 'Unit_Price', 'Discount_Pct']

# Hold out the latest 20% of dates; do not randomly mix dates across the split.
cutoff = df['Date'].quantile(0.80)
train = df[df['Date'] <= cutoff].copy()
test = df[df['Date'] > cutoff].copy()
X_train, y_train = train[features], train[target]
X_test, y_test = test[features], test[target]
print('Training period:', train['Date'].min().date(), 'to', train['Date'].max().date())
print('Testing period:', test['Date'].min().date(), 'to', test['Date'].max().date())
print('Training rows:', len(train), '| Test rows:', len(test))

preprocessor = ColumnTransformer([
    ('categorical', OneHotEncoder(handle_unknown='ignore'), categorical),
    ('numeric', StandardScaler(), numeric)
])

models = {
    'Mean baseline': DummyRegressor(strategy='mean'),
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(max_depth=12, min_samples_leaf=10, random_state=42),
    'Support Vector Regression': SVR(C=10, epsilon=0.1),
    'Neural Network': MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=300, early_stopping=True, random_state=42)
}
results = []
for name, model in models.items():
    fit_X, fit_y = X_train, y_train
    if name == 'Support Vector Regression' and len(fit_X) > 5000:
        sample = fit_X.sample(5000, random_state=42).index
        fit_X, fit_y = fit_X.loc[sample], fit_y.loc[sample]
    pipeline = Pipeline([('preprocessing', preprocessor), ('model', model)])
    pipeline.fit(fit_X, fit_y)
    predictions = pipeline.predict(X_test)
    results.append({
        'Model': name,
        'MAE': mean_absolute_error(y_test, predictions),
        'RMSE': mean_squared_error(y_test, predictions) ** 0.5,
        'R2': r2_score(y_test, predictions)
    })
model_results = pd.DataFrame(results).sort_values('RMSE').reset_index(drop=True)
display(model_results)

## Notes on interpretation

Compare each model with the mean baseline before deciding whether it adds value. The test period is later than the training period, but transaction-level predictions still use attributes available on each row. Results should not be described as future inventory forecasts without adding product-level time lags and stock availability data.

## Power BI handoff

Use the supplied `Revenue`, `Cost` and `Profit` fields for financial reporting. See `dashboard/powerbi_dashboard_spec.md` for the proposed report pages.